In [1]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from scipy.sparse import load_npz
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense,Input,Embedding,SimpleRNN,LSTM,GRU,Bidirectional)
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score)

### import libraries

### Load the datasets

In [2]:
# Phase 2: padded sequences
X_train_padded = np.load("models/X_train_padded.npy")
X_val_padded = np.load("models/X_val_padded.npy")
X_test_padded = np.load("models/X_test_padded.npy")

# Phase 3: TF-IDF
X_train_tfidf = load_npz("models/X_train_tfidf.npz")
X_val_tfidf = load_npz("models/X_val_tfidf.npz")
X_test_tfidf = load_npz("models/X_test_tfidf.npz")

# Labels
y_train = np.load("models/y_train.npy")
y_val = np.load("models/y_val.npy")
y_test = np.load("models/y_test.npy")

In [3]:

print("X_train:", X_train_padded.shape)
print("X_val:", X_val_padded.shape)
print("X_test:", X_test_padded.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)


print("X_train:", X_train_tfidf.shape)
print("X_val:", X_val_tfidf.shape)
print("X_test:", X_test_tfidf.shape)



X_train: (34705, 200)
X_val: (7439, 200)
X_test: (7438, 200)
y_train: (34705,)
y_val: (7439,)
y_test: (7438,)
X_train: (34705, 20000)
X_val: (7439, 20000)
X_test: (7438, 20000)


### Load the Tokenizer

In [4]:
with open("models/tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 85873


In [5]:
sequence_length = X_train_padded.shape[1]
print("Sequence length:", sequence_length)

Sequence length: 200


### ANN model

In [6]:
ann_model = Sequential([
    Input(shape=(X_train_tfidf.shape[1],)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [7]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
ann_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │     2,560,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,568,449 (9.80 MB)

 Trainable params: 2,568,449 (9.80 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history_ann = ann_model.fit(X_train_tfidf,y_train,validation_data=(X_val_tfidf, y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.8732 - loss: 0.3009 - val_accuracy: 0.8953 - val_loss: 0.2551
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9483 - loss: 0.1379 - val_accuracy: 0.8843 - val_loss: 0.2993
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9790 - loss: 0.0586 - val_accuracy: 0.8793 - val_loss: 0.4569
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9918 - loss: 0.0206 - val_accuracy: 0.8777 - val_loss: 0.6293
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9961 - loss: 0.0089 - val_accuracy: 0.8836 - val_loss: 0.7392
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9977 - loss: 0.0058 - val_accuracy: 0.8762 - val_loss: 0.9007
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9985 - loss: 0.0038 - val_accuracy: 0.8770 - val_loss: 1.0218
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9984 - loss: 0

In [9]:
print(history_ann.history.keys())

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])


In [10]:
results = []

## ANN Predictions

In [11]:
y_pred_prob_ann = ann_model.predict(X_test_tfidf)

233/233 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [12]:
y_pred_ann = (y_pred_prob_ann >= 0.5).astype(int).ravel()

In [13]:
ann_accuracy = accuracy_score(y_test, y_pred_ann)
ann_precision = precision_score(y_test,y_pred_ann)
ann_recall = recall_score(y_test,y_pred_ann)
ann_f1 = f1_score(y_test,y_pred_ann)
ann_roc_auc = roc_auc_score(y_test,y_pred_prob_ann.ravel())

In [14]:
print("ANN Results")
print("----------------------")
print("Accuracy :", ann_accuracy)
print("Precision:", ann_precision)
print("Recall   :", ann_recall)
print("F1 Score :", ann_f1)
print("ROC-AUC  :", ann_roc_auc)

ANN Results
----------------------
Accuracy : 0.8693197095993547
Precision: 0.8645365724848165
Recall   : 0.8770425930886686
F1 Score : 0.8707446808510638
ROC-AUC  : 0.9387330346513731


In [15]:
ann_model.save("models/ann_model.keras")

In [16]:
results.append({
    "Model": "ANN",
    "Accuracy": ann_accuracy,
    "Precision": ann_precision,
    "Recall": ann_recall,
    "F1 Score": ann_f1,
    "ROC-AUC": ann_roc_auc
})

### RNN Model

In [17]:
rnn_model = Sequential([
    Input(shape=(sequence_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    SimpleRNN(64),
    Dense(64,activation="relu"),
    Dense(1, activation="sigmoid")
])
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
    )
rnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │    10,991,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,008,321 (41.99 MB)

 Trainable params: 11,008,321 (41.99 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:

history_rnn = rnn_model.fit(X_train_padded,y_train,validation_data=(X_val_padded, y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 54s 49ms/step - accuracy: 0.5034 - loss: 0.6976 - val_accuracy: 0.4878 - val_loss: 0.7092
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 53s 49ms/step - accuracy: 0.5126 - loss: 0.6950 - val_accuracy: 0.4881 - val_loss: 0.6951
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 54s 49ms/step - accuracy: 0.5246 - loss: 0.6808 - val_accuracy: 0.5080 - val_loss: 0.7047
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 53s 49ms/step - accuracy: 0.5613 - loss: 0.6191 - val_accuracy: 0.4997 - val_loss: 0.7552
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 53s 49ms/step - accuracy: 0.5750 - loss: 0.5947 - val_accuracy: 0.5075 - val_loss: 0.8678
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 56s 52ms/step - accuracy: 0.5765 - loss: 0.5906 - val_accuracy: 0.5071 - val_loss: 0.9679
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 54s 50ms/step - accuracy: 0.5752 - loss: 0.5950 - val_accuracy: 0.5015 - val_loss: 0.9225
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 53s 49ms/step - accuracy: 0.5748 -

#### RNN Prediction

In [19]:
y_pred_prob_rnn = rnn_model.predict(X_test_padded)
y_pred_rnn = (y_pred_prob_rnn >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [20]:
rnn_accuracy = accuracy_score(y_test, y_pred_rnn)
rnn_precision = precision_score(y_test,y_pred_rnn)
rnn_recall = recall_score(y_test,y_pred_rnn)
rnn_f1 = f1_score(y_test,y_pred_rnn)
rnn_roc_auc = roc_auc_score(y_test,y_pred_prob_rnn.ravel())

In [21]:
print("RNN Results")
print("----------------------")
print("Accuracy :", rnn_accuracy)
print("Precision:", rnn_precision)
print("Recall   :", rnn_recall)
print("F1 Score :", rnn_f1)
print("ROC-AUC  :", rnn_roc_auc)

RNN Results
----------------------
Accuracy : 0.503226673837053
Precision: 0.5310457516339869
Recall   : 0.08706134476292526
F1 Score : 0.14959723820483314
ROC-AUC  : 0.5025071281306566


In [22]:
rnn_model.save("models/rnn_model.keras")

In [23]:
results.append({
    "Model": "RNN",
    "Accuracy": rnn_accuracy,
    "Precision": rnn_precision,
    "Recall": rnn_recall,
    "F1 Score": rnn_f1,
    "ROC-AUC": rnn_roc_auc
})

### LSTM Model

In [24]:
lstm_model = Sequential([
    Input(shape=(sequence_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    LSTM(64),
    Dense(64,activation="relu"),
    Dense(1, activation="sigmoid")
])
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │    10,991,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,045,377 (42.13 MB)

 Trainable params: 11,045,377 (42.13 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
history_lstm = lstm_model.fit(X_train_padded,y_train,validation_data=(X_val_padded, y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 90s 82ms/step - accuracy: 0.5220 - loss: 0.6875 - val_accuracy: 0.5421 - val_loss: 0.6767
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.7847 - loss: 0.4430 - val_accuracy: 0.8665 - val_loss: 0.3183
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 90s 83ms/step - accuracy: 0.9208 - loss: 0.2152 - val_accuracy: 0.8806 - val_loss: 0.2956
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.9615 - loss: 0.1187 - val_accuracy: 0.8759 - val_loss: 0.3852
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.9800 - loss: 0.0654 - val_accuracy: 0.8719 - val_loss: 0.4318
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.9888 - loss: 0.0383 - val_accuracy: 0.8744 - val_loss: 0.5407
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.9920 - loss: 0.0279 - val_accuracy: 0.8691 - val_loss: 0.5893
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 90s 83ms/step - accuracy: 0.9949 -

### LSTM Prediction

In [26]:
y_pred_prob_lstm = lstm_model.predict(X_test_padded)
y_pred_lstm = (y_pred_prob_lstm >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step


In [27]:

lstm_accuracy = accuracy_score(y_test,y_pred_lstm)
lstm_precision = precision_score(y_test,y_pred_lstm)
lstm_recall = recall_score(y_test,y_pred_lstm)
lstm_f1 = f1_score(y_test,y_pred_lstm)
lstm_roc_auc = roc_auc_score(y_test,y_pred_prob_lstm.ravel())

In [28]:
print("LSTM Results")
print("----------------------")
print("Accuracy :", lstm_accuracy)
print("Precision:", lstm_precision)
print("Recall   :", lstm_recall)
print("F1 Score :", lstm_f1)
print("ROC-AUC  :", lstm_roc_auc)

LSTM Results
----------------------
Accuracy : 0.8628663619252487
Precision: 0.8565045992115637
Recall   : 0.8730243771765336
F1 Score : 0.8646855929954895
ROC-AUC  : 0.9289802118682517


In [29]:
lstm_model.save("models/lstm_model.keras")

In [30]:
results.append({
    "Model": "LSTM",
    "Accuracy": lstm_accuracy,
    "Precision": lstm_precision,
    "Recall": lstm_recall,
    "F1 Score": lstm_f1,
    "ROC-AUC": lstm_roc_auc
})

### GRU Model

In [31]:
gru_model = Sequential([
    Input(shape=(sequence_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    GRU(64),
    Dense(64,activation="relu"),
    Dense(1, activation="sigmoid")
])
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
gru_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 128)       │    10,991,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,033,217 (42.09 MB)

 Trainable params: 11,033,217 (42.09 MB)

 Non-trainable params: 0 (0.00 B)

In [32]:
history_gru = gru_model.fit(X_train_padded,y_train,validation_data=(X_val_padded, y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 81ms/step - accuracy: 0.5063 - loss: 0.6915 - val_accuracy: 0.5179 - val_loss: 0.6853
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.8665 - loss: 0.3132 - val_accuracy: 0.8943 - val_loss: 0.2629
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.9512 - loss: 0.1379 - val_accuracy: 0.8820 - val_loss: 0.3057
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.9821 - loss: 0.0575 - val_accuracy: 0.8750 - val_loss: 0.4232
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.9922 - loss: 0.0247 - val_accuracy: 0.8782 - val_loss: 0.5525
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 82ms/step - accuracy: 0.9969 - loss: 0.0108 - val_accuracy: 0.8778 - val_loss: 0.6469
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.9970 - loss: 0.0098 - val_accuracy: 0.8762 - val_loss: 0.6914
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.9969 -

### GRU Prediction

In [33]:
y_pred_prob_gru = gru_model.predict(X_test_padded)
y_pred_gru = (y_pred_prob_gru >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step


In [34]:
gru_accuracy = accuracy_score(y_test,y_pred_gru)
gru_precision = precision_score(y_test,y_pred_gru)
gru_recall = recall_score(y_test,y_pred_gru)
gru_f1 = f1_score(y_test,y_pred_gru)
gru_roc_auc = roc_auc_score(y_test,y_pred_prob_gru.ravel())

In [35]:
print("GRU Results")
print("----------------------")
print("Accuracy :", gru_accuracy)
print("Precision:", gru_precision)
print("Recall   :", gru_recall)
print("F1 Score :", gru_f1)
print("ROC-AUC  :", gru_roc_auc)

GRU Results
----------------------
Accuracy : 0.8759075020166711
Precision: 0.885989010989011
Recall   : 0.8639164211090276
F1 Score : 0.8748135087481351
ROC-AUC  : 0.9401578654542969


In [36]:
gru_model.save("models/gru_model.keras")

In [37]:
results.append({
    "Model": "GRU",
    "Accuracy": gru_accuracy,
    "Precision": gru_precision,
    "Recall": gru_recall,
    "F1 Score": gru_f1,
    "ROC-AUC": gru_roc_auc
})

### Bidirectional LSTM

In [38]:
bilstm_model = Sequential([
    Input(shape=(sequence_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    Bidirectional(
        LSTM(64)
    ),
    Dense(64,activation="relu"),
    Dense(1, activation="sigmoid")
])
bilstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
bilstm_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │    10,991,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,098,881 (42.34 MB)

 Trainable params: 11,098,881 (42.34 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
history_bilstm = bilstm_model.fit(X_train_padded,y_train,validation_data=(X_val_padded, y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 144s 132ms/step - accuracy: 0.8426 - loss: 0.3634 - val_accuracy: 0.8835 - val_loss: 0.2852
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 150s 138ms/step - accuracy: 0.9234 - loss: 0.2055 - val_accuracy: 0.8720 - val_loss: 0.3266
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 149s 138ms/step - accuracy: 0.9543 - loss: 0.1315 - val_accuracy: 0.8843 - val_loss: 0.3372
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 149s 137ms/step - accuracy: 0.9689 - loss: 0.0935 - val_accuracy: 0.8805 - val_loss: 0.4286
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 151s 139ms/step - accuracy: 0.9823 - loss: 0.0546 - val_accuracy: 0.8769 - val_loss: 0.5001
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 150s 138ms/step - accuracy: 0.9865 - loss: 0.0407 - val_accuracy: 0.8774 - val_loss: 0.5410
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 149s 138ms/step - accuracy: 0.9914 - loss: 0.0267 - val_accuracy: 0.8691 - val_loss: 0.5895
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 150s 138ms/step - ac

### Bidirectional LSTM

In [40]:
y_pred_prob_bilstm = bilstm_model.predict(X_test_padded)
y_pred_bilstm = (y_pred_prob_bilstm >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step


In [41]:
bilstm_accuracy = accuracy_score(y_test,y_pred_bilstm)
bilstm_precision = precision_score(y_test,y_pred_bilstm)
bilstm_recall = recall_score(y_test,y_pred_bilstm)
bilstm_f1 = f1_score(y_test,y_pred_bilstm)
bilstm_roc_auc = roc_auc_score(y_test,y_pred_prob_bilstm.ravel())

In [42]:
print("Bi-LSTM Results")
print("----------------------")
print("Accuracy :", bilstm_accuracy)
print("Precision:", bilstm_precision)
print("Recall   :", bilstm_recall)
print("F1 Score :", bilstm_f1)
print("ROC-AUC  :", bilstm_roc_auc)

Bi-LSTM Results
----------------------
Accuracy : 0.8726808281796182
Precision: 0.8560838445807771
Recall   : 0.8971336726493437
F1 Score : 0.8761281883584042
ROC-AUC  : 0.9292816051751295


In [43]:
bilstm_model.save("models/bilstm_model.keras")

In [44]:
results.append({
    "Model": "Bi-LSTM",
    "Accuracy": bilstm_accuracy,
    "Precision": bilstm_precision,
    "Recall": bilstm_recall,
    "F1 Score": bilstm_f1,
    "ROC-AUC": bilstm_roc_auc
})

In [45]:
comparison_df = pd.DataFrame(results)
comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN,0.869320,0.864537,0.877043,0.870745,0.938733
1,RNN,0.503227,0.531046,0.087061,0.149597,0.502507
2,LSTM,0.862866,0.856505,0.873024,0.864686,0.928980
3,GRU,0.875908,0.885989,0.863916,0.874814,0.940158
4,Bi-LSTM,0.872681,0.856084,0.897134,0.876128,0.929282


In [46]:
comparison_df.to_csv("models/phase4_model_comparison.csv",index=False)